# SiamFC++：Google Colab 測試環境（修正版）

這份 Notebook 是可重現的環境準備與測試流程骨架。SiamFC++ 的相依套件、CUDA/PyTorch 版本及模型權重會隨原始碼版本而異，因此先檢查 GPU、下載原始碼，再依專案實際設定安裝；**尚未宣稱已在 Colab 完成端到端追蹤測試**。

參考文章：<https://blog.csdn.net/qq_30347421/article/details/104534297>

> 建議：在 Colab 選「執行階段 → 變更執行階段類型 → T4 GPU」，然後由上而下執行。若原始碼倉庫位置或啟動命令已變更，請依下載到的 README 修正對應步驟。


## 1. 確認執行環境


In [ ]:
import os, sys, platform
print('Python:', sys.version)
print('Platform:', platform.platform())
try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('PyTorch 尚未安裝或載入失敗：', repr(exc))


## 2. 下載原始碼

預設嘗試常見的 SiamFC++/VideoAnalyst 原始碼倉庫。若 `git clone` 回報倉庫不存在或無法連線，請先確認網路、GitHub 可用性，以及文章指定的實際倉庫網址。

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = 'https://github.com/MegviiDetection/video_analyst.git'
WORKDIR = Path('/content/video_analyst')

if WORKDIR.exists():
    print(f'資料夾已存在：{WORKDIR}；略過 clone。')
else:
    result = subprocess.run(
        ['git', 'clone', '--depth', '1', REPO_URL, str(WORKDIR)],
        text=True, capture_output=True
    )
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError('Git clone 失敗。請確認網路與倉庫網址，再重新執行本節。')

print('Repository exists:', WORKDIR.exists())
if WORKDIR.exists():
    print('\n'.join(str(p.relative_to(WORKDIR)) for p in list(WORKDIR.iterdir())[:40]))


## 3. 查看專案安裝說明與測試入口

先讀取倉庫中的 README、環境設定及測試腳本，避免硬套與目前版本不相容的舊指令。

In [ ]:
from pathlib import Path
WORKDIR = Path('/content/video_analyst')
if not WORKDIR.exists():
    raise FileNotFoundError('請先成功執行第 2 節。')

candidates = []
for pattern in ('README*', '*requirements*.txt', 'environment*.yml', 'environment*.yaml', 'setup.py', 'pyproject.toml'):
    candidates.extend(WORKDIR.glob(pattern))
for path in candidates[:20]:
    print(f'\\n===== {path.name} =====')
    try:
        print(path.read_text(encoding='utf-8', errors='replace')[:6000])
    except Exception as exc:
        print('讀取失敗：', exc)

print('\\n可能的測試入口：')
for path in WORKDIR.rglob('*.py'):
    if any(word in path.name.lower() for word in ('test', 'demo', 'eval', 'track')):
        print(path.relative_to(WORKDIR))


## 4. 檢查模型權重與設定檔

SiamFC++ 通常需要與模型設定相符的權重檔。請依原始碼 README 下載官方權重，並將路徑填入下方；不要使用來源不明的權重。

In [ ]:
from pathlib import Path

# 請依原始碼說明修改成實際的權重與設定檔路徑。
CHECK_PATHS = [
    Path('/content/video_analyst'),
    # Path('/content/video_analyst/實際權重檔.pth'),
    # Path('/content/video_analyst/實際設定檔.yaml'),
]
for p in CHECK_PATHS:
    print(('OK      ' if p.exists() else 'MISSING '), p)

weight_suffixes = {'.pth', '.pt', '.ckpt', '.pkl'}
config_suffixes = {'.yaml', '.yml', '.json'}
repo = Path('/content/video_analyst')
if repo.exists():
    weights = [p for p in repo.rglob('*') if p.is_file() and p.suffix.lower() in weight_suffixes]
    configs = [p for p in repo.rglob('*') if p.is_file() and p.suffix.lower() in config_suffixes]
    print(f'找到權重檔數量：{len(weights)}')
    print('權重範例：', [str(p.relative_to(repo)) for p in weights[:20]])
    print(f'找到設定檔數量：{len(configs)}')
    print('設定檔範例：', [str(p.relative_to(repo)) for p in configs[:20]])


## 5. 執行實際測試（需依倉庫版本設定）

此處刻意不猜測舊版 `main/test.py` 的參數。先執行下列指令查看腳本說明；若腳本不存在，請根據第 3 節列出的入口與 README 調整。

In [ ]:
from pathlib import Path
import subprocess

repo = Path('/content/video_analyst')
possible_scripts = [repo / 'main' / 'test.py', repo / 'test.py']
test_script = next((p for p in possible_scripts if p.exists()), None)
if test_script is None:
    print('沒有找到預期的 main/test.py 或 test.py。請查看第 3 節列出的入口與 README。')
else:
    print('找到測試腳本：', test_script)
    result = subprocess.run([sys.executable, str(test_script), '--help'], cwd=str(repo), text=True, capture_output=True)
    print('return code:', result.returncode)
    print(result.stdout[:10000])
    print(result.stderr[:5000])
    if result.returncode != 0:
        print('此專案可能不支援 --help，請依 README 的正式測試命令執行。')


## 6. 執行前檢查清單

- [ ] GPU 已啟用且 PyTorch 可辨識 CUDA（若預期使用 GPU）。
- [ ] 原始碼成功下載，並確認 README 對應的版本。
- [ ] 已安裝該版本指定的依賴套件。
- [ ] 已取得官方模型權重，並確認設定檔與權重配對。
- [ ] 測試影片／資料集路徑正確。
- [ ] 已依 README 使用正確參數執行追蹤測試。

**注意：** 這份檔案已做 Notebook 結構與 Python 語法檢查，但因無法在此環境連線下載 GitHub、安裝完整依賴並取得權重，不能把它描述為已通過 Colab 端到端測試。